# Phase 1: Cleaning the Telco churn dataset, step by step

This notebook does exactly what the **Telco** part of `clean_data.py` does, but slowly, one small step at a time, with an explanation before every cell.

**How to use it:** click a cell and press **Shift + Enter** to run it. Read the text above each cell first, then look at what it prints. Run the cells from top to bottom, in order.

**The plan**

| Part | What we do |
|---|---|
| A | Load the data and take a first look |
| B | Check data quality: missing values, duplicates, wrong formats, inconsistent labels, outliers, class imbalance |
| C | Fix each problem, and say why we chose that fix |
| D | Final checks and save the clean file |

**The dataset:** IBM Telco Customer Churn (Kaggle, user *blastchar*). One row = one customer of a phone and internet company. The column **Churn** says whether the customer left (Yes) or stayed (No). That is what we will later try to predict.

---
# Part A: Load and look

## Step 1. Import the libraries

A *library* is a toolbox someone else wrote.
- **pandas** (we call it `pd`) works with tables. A table in pandas is called a **DataFrame**.
- **numpy** (`np`) does maths on numbers.
- **matplotlib** (`plt`) draws charts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Step 2. Load the raw file

`pd.read_csv(...)` reads a CSV file (a text file where values are separated by commas) into a DataFrame. We store it in a variable called `t` (for Telco), the same name the script uses.

The path `data_raw/telco.csv` is relative to this notebook, so the file sits in the `data_raw` folder next to it.

In [ ]:
t = pd.read_csv("data_raw/telco.csv")
t.shape

`shape` gives **(rows, columns)**. We have 7,043 customers and 21 columns.

## Step 3. Look at the first rows

`head()` shows the first 5 rows. Always do this first: it shows you what the data really looks like.

In [ ]:
t.head()

What to notice:
- `customerID` is just a label like `7590-VHVEG`.
- Many columns hold text such as `Yes`, `No`, `No phone service`.
- `tenure` is how many months the person has been a customer.
- `MonthlyCharges` is the monthly bill; `TotalCharges` is everything they have paid so far.

## Step 4. Check the type of each column

Every column has a **data type** (dtype):
- `int64` = whole numbers, `float64` = numbers with decimals
- `object` = text

A number column stored as text is a **format problem**: you cannot calculate with it.

In [ ]:
t.dtypes

**Problem found:** `TotalCharges` is `object` (text), but it should be a number. Something inside it is not a number. We will find out what in Step 6.

---
# Part B: Data quality checks

## Step 5. Missing values (the normal way)

`isna()` marks every empty cell as True. `.sum()` counts them per column.

In [ ]:
t.isna().sum()

All zeros. It looks like nothing is missing... but Step 4 told us `TotalCharges` has something strange in it. Let's look closer.

## Step 6. Hidden missing values

Sometimes a "missing" value is not truly empty. It is a **blank space** `" "`. pandas sees a space as normal text, so `isna()` does not catch it.

`.str.strip()` removes spaces from the start and end of the text. If what remains is `""` (nothing), the cell was blank.

In [ ]:
blank = t["TotalCharges"].str.strip() == ""
blank.sum()

**Problem found:** 11 customers have a blank `TotalCharges`. `blank` is now a list of True/False, one per row. We can use it to show just those rows:

In [ ]:
t.loc[blank, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

Look at `tenure`: it is **0 for all 11**. These are brand-new customers who joined this month. They have not received a bill yet, so nothing has been charged in total.

This is the key insight: the value is not "unknown", it is really **zero**. Let's confirm it with code. `assert` stops with an error if the statement is false; if it prints nothing, it is true.

In [ ]:
assert (t.loc[blank, "tenure"] == 0).all()
print("Confirmed: every blank TotalCharges belongs to a customer with tenure 0")

## Step 7. Duplicates

A duplicate is the same row appearing twice. `duplicated()` marks repeated rows; `.sum()` counts them.

In [ ]:
print("Fully duplicated rows:", t.duplicated().sum())
print("Repeated customer IDs:", t["customerID"].duplicated().sum())

No duplicates, and every customer ID is unique. One more check: if we ignore the ID, are there customers with exactly the same answers in every column?

In [ ]:
t.drop(columns="customerID").duplicated().sum()

22 rows look the same as another row once the ID is hidden. **We keep them.** They have different IDs, so they are different people. With only a few yes/no columns, two new customers on the same cheap plan can easily look identical. Deleting them would remove real customers.

## Step 8. Inconsistent labels in the text columns

For text columns we look at which values appear and how often, using `value_counts()`. First, let's list the text columns:

In [ ]:
text_cols = t.select_dtypes(include="object").columns.drop(["customerID", "TotalCharges"])
list(text_cols)

In [ ]:
for c in text_cols:
    counts = t[c].value_counts()
    print(c, ":", ", ".join(f"{k} = {v}" for k, v in counts.items()))

Read the output slowly. Two things to notice:

1. **Spelling is clean.** No `yes` vs `Yes`, no extra spaces. Good.
2. **Some columns have a third answer that means "No" in disguise.** For example `OnlineSecurity` can be `Yes`, `No`, or `No internet service`. A customer with no internet obviously does not have online security either. The same happens with `MultipleLines` and `No phone service`.

Is the "No internet service" information lost if we turn it into "No"? Let's check that the `InternetService` column already tells us who has no internet:

In [ ]:
no_net = t["InternetService"] == "No"
print("Customers with InternetService = No:", no_net.sum())
print("Customers with OnlineSecurity = 'No internet service':", (t["OnlineSecurity"] == "No internet service").sum())
print("Same people?", (no_net == (t["OnlineSecurity"] == "No internet service")).all())

Same 1,526 people. So the information is already stored once in `InternetService`. Repeating it in 6 more columns adds nothing. (The same is true for `PhoneService` and `No phone service`.)

## Step 9. Outliers in the number columns

An **outlier** is a value far away from the others. It could be a typing error (e.g. tenure = 900 months) or just a rare, real customer.

`describe()` gives a quick summary: count, mean, min, max and the quartiles (25%, 50% = median, 75%).

In [ ]:
num_cols = ["tenure", "MonthlyCharges"]
t[num_cols].describe()

These ranges make sense: tenure is 0 to 72 months (6 years), monthly charges are about \$18 to \$119. Nothing impossible.

A common rule from statistics is the **IQR rule**: IQR = Q3 − Q1. Any value below Q1 − 1.5×IQR or above Q3 + 1.5×IQR is called an outlier. Let's count them:

In [ ]:
for c in num_cols:
    q1, q3 = t[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((t[c] < low) | (t[c] > high)).sum()
    print(f"{c}: allowed range {low:.1f} to {high:.1f}, outliers = {n_out}")

Zero outliers. A box plot shows the same thing as a picture: the "whiskers" reach the min and max, with no dots beyond them.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, c in zip(axes, num_cols):
    ax.boxplot(t[c], vert=False)
    ax.set_title(c)
plt.tight_layout()
plt.show()

## Step 10. Class imbalance in the target

The **target** is the column we want to predict: `Churn`. If one answer is much more common than the other, the data is **imbalanced**.

`value_counts(normalize=True)` gives proportions instead of counts.

In [ ]:
print(t["Churn"].value_counts())
print(t["Churn"].value_counts(normalize=True).round(3))

In [ ]:
t["Churn"].value_counts().plot(kind="bar", title="Churn: how many customers left?")
plt.ylabel("customers")
plt.show()

**Problem found:** about 73% stayed and 27% left. Why it matters: a lazy model that always says "No" would already be 73% correct, while catching zero churners. So later we must judge models with measures like recall, F1 or ROC-AUC, not just accuracy.

We do **not** fix this now. See Step 17 for why.

### Summary of what we found

| Check | Result |
|---|---|
| Format | `TotalCharges` stored as text |
| Missing | 11 hidden blanks in `TotalCharges` (all tenure = 0) |
| Duplicates | none (22 look-alike profiles, kept) |
| Labels | "No internet service" / "No phone service" repeat what other columns say |
| Outliers | none by the IQR rule |
| Imbalance | 73% No / 27% Yes |

---
# Part C: Fix each problem

From here on, every cell matches a line in `clean_data.py`. We keep the original `t` untouched for comparison, and make a copy called `clean` to change.

In [ ]:
clean = t.copy()

## Step 11. Fix TotalCharges: text → number, blanks → 0

`pd.to_numeric(..., errors="coerce")` turns text into numbers. *coerce* means "if you can't convert it, put NaN (missing) instead of crashing". So the 11 blanks become NaN. Then `.fillna(0.0)` replaces those NaN with 0.

**Why 0 and not the average?** Step 6 showed these customers have tenure 0, so they have truly paid nothing yet. Filling with the average (about \$2,280) would invent money they never paid. Deleting the rows would also work (only 11 of 7,043), but 0 is the *correct* value, so we keep them.

In [ ]:
clean["TotalCharges"] = pd.to_numeric(clean["TotalCharges"], errors="coerce").fillna(0.0)
print(clean["TotalCharges"].dtype, "| missing:", clean["TotalCharges"].isna().sum())

A quick sanity check: total paid should be roughly months × monthly bill. The correlation (a number from −1 to 1) should be close to 1.

In [ ]:
approx = clean["tenure"] * clean["MonthlyCharges"]
print("Correlation:", round(np.corrcoef(approx, clean["TotalCharges"])[0, 1], 3))

## Step 12. Drop customerID

**Why?** An ID is just a name tag. It is unique for every person, so it carries no pattern a model could learn from. Worse, a model could "memorise" IDs and look good for the wrong reason. We checked in Step 7 that it has no duplicates, so its job is done.

In [ ]:
clean = clean.drop(columns="customerID")
clean.shape

## Step 13. Merge "No internet service" / "No phone service" into "No"

`.replace({old: new})` swaps values. We do this for the 7 service columns.

**Why?** Step 8 showed the "no internet" information already lives in `InternetService` (and "no phone" in `PhoneService`). After the merge each of these columns becomes a simple Yes/No question: *does this customer have this service?*

In [ ]:
service_cols = ["MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
                "TechSupport", "StreamingTV", "StreamingMovies"]
for c in service_cols:
    clean[c] = clean[c].replace({"No internet service": "No", "No phone service": "No"})

clean["OnlineSecurity"].value_counts()

## Step 14. Turn Yes/No into 1/0

Models work with numbers, not words. `.map({"Yes": 1, "No": 0})` swaps each word for a number.

**Why 1 and 0?** For a two-answer column this is all you need: 1 = has it, 0 = doesn't. The mean of the column then equals the share of "Yes", which is handy (e.g. the mean of `Churn` = churn rate).

In [ ]:
yes_no_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"] + service_cols
for c in yes_no_cols:
    clean[c] = clean[c].map({"Yes": 1, "No": 0})

# check that nothing became NaN (that would mean an unexpected word)
clean[yes_no_cols].isna().sum().sum()

0 means every value was either Yes or No, so nothing was lost.

## Step 15. Turn gender into IsMale (1/0)

Same idea for gender: Male → 1, Female → 0. We rename the column to `IsMale`, so the name tells you what 1 means.

In [ ]:
clean["gender"] = clean["gender"].map({"Male": 1, "Female": 0})
clean = clean.rename(columns={"gender": "IsMale"})
clean["IsMale"].value_counts()

## Step 16. Outliers: no fix needed

Step 9 found no outliers, so we change nothing. Writing this down still matters: in your report you show that you **checked**, and explain why you left the values alone.

## Step 17. What we deliberately do NOT do here

Three columns still contain text with more than two options: `InternetService`, `Contract`, `PaymentMethod`. The numbers are also on very different scales (tenure up to 72, TotalCharges up to about 8,700). And the target is imbalanced.

The fixes for these are **one-hot encoding**, **scaling**, and **resampling** (e.g. SMOTE). We leave them for Phase 2. Why?

> These fixes *learn something from the data* (the mean for scaling, new fake rows for SMOTE). If we learn it from the whole dataset, information from the future **test** rows leaks into training, and our results look better than they really are. This is called **data leakage**.

So in Phase 2 we will first split into train and test, and then do these steps inside a scikit-learn `Pipeline` that only learns from the training part. That is what the note at the top of `clean_data.py` means.

In [ ]:
print(clean.select_dtypes(include="object").columns.tolist())

---
# Part D: Final checks and save

## Step 18. Final check

In [ ]:
print("Shape:", clean.shape)
print("Missing values:", clean.isna().sum().sum())
print("Churn rate:", round(clean["Churn"].mean(), 3))
clean.head()

7,043 rows (we lost no customers), 20 columns (we dropped only the ID), no missing values, churn rate 26.5%.

## Step 19. Save and compare with the script's output

`to_csv(..., index=False)` writes the table to a file. `index=False` stops pandas adding an extra column of row numbers.

We save under a new name so we don't overwrite the file the script made. Then we load the script's file and check the two are identical.

In [ ]:
clean.to_csv("data_clean/telco_clean_notebook.csv", index=False)

script_version = pd.read_csv("data_clean/telco_clean.csv")
notebook_version = pd.read_csv("data_clean/telco_clean_notebook.csv")
print("Identical to clean_data.py output:", script_version.equals(notebook_version))

---
# Recap: how this notebook maps to `clean_data.py`

| Script line | Notebook step | Why |
|---|---|---|
| `t = pd.read_csv(...)` | 2 | Load the raw data |
| `blank = t.TotalCharges.str.strip() == ""` | 6 | Find the 11 hidden blanks |
| `assert (t.loc[blank, "tenure"] == 0).all()` | 6 | Prove they are all new customers |
| `pd.to_numeric(..., errors="coerce").fillna(0.0)` | 11 | Text → number; blank → 0 because nothing was billed yet |
| `t.drop(columns="customerID")` | 12 | An ID has no pattern to learn |
| `.replace({"No internet service": "No", ...})` | 13 | Same info already in InternetService / PhoneService |
| `.map({"Yes": 1, "No": 0})` | 14 | Models need numbers |
| `gender → IsMale` | 15 | Same, with a clear name |
| *(nothing)* | 16, 17 | No outliers; scaling, one-hot and resampling wait for Phase 2 to avoid leakage |
| `t.to_csv(...)` | 19 | Save the clean file |

The Bank and Iran parts of the script follow the same pattern: load, check, fix with a reason, save.